# 06 — Async Streaming: Examples

> Run multiple LLM streams concurrently with asyncio.gather.

**What you'll build:**
1. Single async stream
2. Parallel streams (asyncio.gather)
3. Async stream with timeout
4. Queue-based token producer/consumer

In [ ]:
import os, time, asyncio
from openai import AsyncOpenAI
from dotenv import load_dotenv
from rich import print as rprint
from rich.table import Table
from rich.console import Console

load_dotenv()
async_client = AsyncOpenAI()
console = Console()
print('✅ Setup complete')

---
## Part 1: Single Async Stream

In [ ]:
async def async_stream(prompt: str, model: str = "gpt-4o-mini") -> dict:
    start = time.perf_counter()
    ttft = None
    text = ""

    stream = await async_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True, max_tokens=100
    )

    async for chunk in stream:  # ← async for
        c = chunk.choices[0].delta.content or ""
        if c:
            if ttft is None: ttft = time.perf_counter()
            text += c
            print(c, end="", flush=True)

    print()
    total = time.perf_counter() - start
    return {"text": text, "ttft_ms": round((ttft-start)*1000, 1) if ttft else None, "total_ms": round(total*1000, 1)}


print("🟢 Single async stream:\n")
result = await async_stream("What is asyncio? Answer in one sentence.")
rprint(f"\nTTFT: {result['ttft_ms']}ms | Total: {result['total_ms']}ms")

---
## Part 2: Parallel Streams with asyncio.gather

In [ ]:
async def stream_labeled(prompt: str, label: str, q: asyncio.Queue) -> dict:
    """Stream one prompt and put labeled tokens into queue."""
    start = time.perf_counter()
    text = ""
    stream = await async_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        stream=True, max_tokens=60
    )
    async for chunk in stream:
        c = chunk.choices[0].delta.content or ""
        if c:
            text += c
            await q.put((label, c))
    await q.put((label, None))  # Sentinel
    return {"label": label, "text": text, "ms": round((time.perf_counter()-start)*1000, 1)}


async def run_parallel_streams(prompts_and_labels: list[tuple[str, str]]):
    q = asyncio.Queue()

    # Display consumer — print tokens as they arrive
    finished = {label: False for _, label in prompts_and_labels}

    async def display():
        buffers = {label: "" for _, label in prompts_and_labels}
        active = len(prompts_and_labels)
        while active > 0:
            label, token = await q.get()
            if token is None:
                active -= 1
            else:
                buffers[label] += token

    # Producer tasks
    producer_tasks = [stream_labeled(p, l, q) for p, l in prompts_and_labels]

    # Run producers and display concurrently
    results, _ = await asyncio.gather(
        asyncio.gather(*producer_tasks),
        display()
    )
    return results


prompts = [
    ("What is Python? One sentence.",    "python"),
    ("What is asyncio? One sentence.",   "asyncio"),
    ("What is streaming? One sentence.", "streaming"),
    ("What is an LLM? One sentence.",    "llm"),
]

print("🚀 Running 4 streams in parallel...\n")
wall_start = time.perf_counter()
results = await run_parallel_streams(prompts)
wall_ms = (time.perf_counter() - wall_start) * 1000

table = Table(title="Parallel Streaming Results", show_header=True)
table.add_column("Label", style="cyan")
table.add_column("Individual Time", style="yellow")
table.add_column("Response", style="white", max_width=60)

for r in results:
    table.add_row(r['label'], f"{r['ms']}ms", r['text'][:58]+".." if len(r['text'])>58 else r['text'])

console.print(table)
rprint(f"\n⏱️  Wall time for all 4: [bold green]{wall_ms:.0f}ms[/bold green] (vs ~{sum(r['ms'] for r in results):.0f}ms sequential)")

---
## Part 3: Async Stream with Timeout

In [ ]:
async def stream_with_timeout(prompt: str, timeout_s: float = 5.0) -> dict:
    """Stream with a hard timeout. Returns partial response if exceeded."""
    text = ""
    timed_out = False

    async def _do_stream():
        nonlocal text
        stream = await async_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            stream=True
        )
        async for chunk in stream:
            c = chunk.choices[0].delta.content or ""
            text += c
            print(c, end="", flush=True)

    try:
        await asyncio.wait_for(_do_stream(), timeout=timeout_s)
    except asyncio.TimeoutError:
        timed_out = True
        rprint(f"\n   [red]⚠️ Timeout after {timeout_s}s[/red]")

    print()
    return {"text": text, "timed_out": timed_out}


print("⏱️  Testing with generous timeout (10s):")
r = await stream_with_timeout("Explain quantum computing in 2 sentences.", timeout_s=10.0)
rprint(f"   Completed: {not r['timed_out']} | Length: {len(r['text'])} chars")

---
## Summary

| Feature | Async API |
|---|---|
| Client | `AsyncOpenAI()` |
| Stream | `await client.chat.completions.create(..., stream=True)` |
| Iterate | `async for chunk in stream:` |
| Parallel | `await asyncio.gather(stream_a(), stream_b())` |
| Timeout | `await asyncio.wait_for(coro, timeout=N)` |

**Next:** `07_production_streaming` — FastAPI SSE endpoint for production.